# 1. 라이브러리 설치
* 가상환경 설정 후 requirements.txt 설치 진행
  `pip install -r requirements.txt`

# 2. 필요한 라이브러리 import

In [1]:
from gensim.models.word2vec import Word2Vec
from kiwipiepy import Kiwi  # 형태소 분석기
# 문자열 정리를 위해 정규 표현식 라이브러리 사용
import re 

# 3. 데이터 수집

- 대한민국 헌법.txt 파일 업로드

In [2]:
with open("대한민국헌법.txt", encoding='cp949') as f:
	content = f.read()

print(len(content)) # 19063글자로 이루어진 문서

19063


# 4. 데이터 전처리 - 문장에서 불필요한 문자 제거

In [7]:
# 데이터 정리
print("[정리 전]\n", content[:100])

# 문장별로 분리하기 (한글 기준, '.', '?', '!' 뒤에 공백이 오거나 개행이 오는 경우 분리)
docs = re.split(r'(?<=[.?!])\s+', content.strip())

# 정규 표현식을 통한 한글 외 문자 제거
docs = [re.sub("[^가-힣 ]", "", doc) for doc in docs]  # 완성형 한글(가-힣)만 유지
print("-"*50)
print("[정리 후]\n", docs[:50]) # 정리 후 5개 문장 확인

[정리 전]
 
대한민국헌법
[시행 1988. 2. 25.] [헌법 제10호, 1987. 10. 29., 전부개정]

  전문
  유구한 역사와 전통에 빛나는 우리 대한국민은 3ㆍ1운동으로 건립
--------------------------------------------------
[정리 후]
 ['대한민국헌법시행 ', '', ' 헌법 제호 ', '', ' 전부개정  전문  유구한 역사와 전통에 빛나는 우리 대한국민은 운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한 민주이념을 계승하고 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의인도와 동포애로써 민족의 단결을 공고히 하고 모든 사회적 폐습과 불의를 타파하며 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고히 하여 정치경제사회문화의 모든 영역에 있어서 각인의 기회를 균등히 하고 능력을 최고도로 발휘하게 하며 자유와 권리에 따르는 책임과 의무를 완수하게 하여 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다짐하면서 년 월 일에 제정되고 차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정한다', '년 월 일       제장 총강 제조 대한민국은 민주공화국이다', '대한민국의 주권은 국민에게 있고 모든 권력은 국민으로부터 나온다', '제조 대한민국의 국민이 되는 요건은 법률로 정한다', '국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다', '제조 대한민국의 영토는 한반도와 그 부속도서로 한다', '제조 대한민국은 통일을 지향하며 자유민주적 기본질서에 입각한 평화적 통일정책을 수립하고 이를 추진한다', '제조 대한민국은 국제평화의 유지에 노력하고 침략적 전쟁을 부인한다', '국군은 국가의 안전보장과 국토방위의 신성한 의무를 수행함을 사명으로 하며 그 정치적 중립성은 준수된다', '제조 헌법에 의하여 체결공포된 조약과 일반적으로 승인된 국제법규는 

# 5. 데이터 전처리 - 문장을 단어로 분리(토큰화)

In [4]:
# 불용어(분석에 도움이 되지 않는 단어들) 목록을 미리 정의
# 예: 조사(의, 가, 이, 는, 등), 자주 등장하지만 의미가 적은 단어(좀, 잘 등)
stopwords = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']

In [19]:
# 형태소 분석기 Kiwi를 사용하여 문장들을 형태소(단어 단위)로 나눌 준비
kiwi = Kiwi()

# 최종 토큰(단어)들을 모아둘 리스트
tokenized_data = []
# 각 문장(docs의 원소)에 대해 반복하면서 토큰으로 만듦
for sentence in (docs):
    tokenized_sentence = [token.form for token in kiwi.tokenize(sentence)] 

    # 불용어 목록에 있는 단어들을 제거
    sentence = [word for word in tokenized_sentence if word not in stopwords]
    # 정제된 단어 리스트를 tokenized_data에 추가
    tokenized_data.append(sentence)
print(tokenized_data)

[['대한민국', '헌법', '시행'], [], ['헌법', '제호'], [], ['전부', '개정', '전문', '유구', '하', 'ᆫ', '역사', '전통', '빛나', '우리', '대한', '국민', '운동', '건립', '되', 'ᆫ', '대한민국', '임시', '정부', '법통', '불의', '항거', '하', 'ᆫ', '민주', '이념', '계승', '하', '고', '조국', '민주', '개혁', '평화', '적', '통일', '사명', '입각', '하', '어', '정의', '인도', '동포애', '로써', '민족', '단결', '공고히', '하', '고', '모든', '사회', '적', '폐습', '불의', '타파', '하', '며', '자율', '조화', '바탕', '자유', '민주', '적', '기본', '질서', '더욱', '확고히', '하', '어', '정치', '경제', '사회', '문화', '모든', '영역', '있', '어서', '각', '인', '기회', '균등', '히', '하', '고', '능력', '최', '고도', '로', '발휘', '하', '게', '하', '며', '자유', '권리', '따르', '책임', '의무', '완수', '하', '게', '하', '어', '안', '국민', '생활', '균등', '하', 'ᆫ', '향상', '기하', '고', '밖', '항구', '적', 'ᆫ', '세계', '평화', '인류', '공영', '이바지', '하', 'ᆷ', '으로써', '우리', '우리', '자손', '안전', '자유', '행복', '영원히', '확보', '하', 'ᆯ', '것', '다짐', '하', '면서', '년', '월', '일', '제정', '되', '고', '차', '걸치', '어', '개정', '되', 'ᆫ', '헌법', '이제', '국회', '의결', '거치', '어', '국민', '투표', '의하', '어', '개정', '하', 'ᆫ다'], ['년', '월', '일', '제장', '총강', '제조',

# 6. Word2Vec Model을 활용
- 한글 토큰을 100차원의 워드 임베딩 벡터로 변환

In [8]:
# sentences: 학습에 사용할 토큰화된 문장들
# vector_size: 단어 벡터 차원 크기(100차원)
# window: 주변 단어를 고려하는 윈도우 크기(5)
# min_count: 최소 등장 횟수(1회 미만이면 무시)
# workers: 병렬 작업 쓰레드 수(4)
# sg: 0이면 CBOW 방식, 1이면 Skip-Gram 방식
model = Word2Vec(
    sentences = tokenized_data,
    vector_size = 100,
    window = 5,
    min_count = 1,
    workers = 4,
    sg = 0
)

In [21]:

# 가장 많은 빈도수를 보이는 단어 50개 출력
print(model.wv.index_to_key[:50])

['제조', '법률', '있', '정하', '수', '대통령', '의하', '국회', '국가', '국민', '헌법', '때', '관하', '하', '바', '의원', '위하', '가지', '선거', '없', '필요', '위원', '기타', '법원', '국무', '사항', '보장', '정부', '회의', '법관', '권리', '자유', '일', '의무', '경제', '임명', '직무', '대하', '국무총리', '임기', '조직', '이상', '경우', '공무원', '의결', '범위', '단체', '행정', '정당', '보호']


# 7. 불용어 정의 업데이트 및 코드 재 실행 후, 결과 확인​ 

In [26]:
# '을' 추가
stopwords = ['을', '의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']
# 유지할 품사 태그: NNG(일반명사), NNP(고유명사), NNB(의존명사), VV(동사), VA(형용사)
keep_tags = {'NNG', 'NNP', 'NNB', 'VV', 'VA'}

tokenized_data = []
for sentence in (docs):
    # 유지할 품사만 선택
    tokenized_sentence = [token.form for token in kiwi.tokenize(sentence)
                          if token.tag in keep_tags]
    sentence = [word for word in tokenized_sentence if word not in stopwords]
    tokenized_data.append(sentence)

model = Word2Vec(
    sentences = tokenized_data,
    vector_size = 100,
    window = 5,
    min_count = 1,
    workers = 4,
    sg = 0
)

In [24]:
# 가장 많은 빈도수를 보이는 단어 50개 출력
print(model.wv.index_to_key[:50])

['제조', '법률', '있', '정하', '수', '대통령', '의하', '국회', '국가', '국민', '헌법', '때', '관하', '하', '바', '의원', '위하', '가지', '선거', '없', '필요', '위원', '기타', '법원', '국무', '사항', '보장', '정부', '회의', '법관', '권리', '자유', '일', '의무', '경제', '임명', '직무', '대하', '국무총리', '임기', '조직', '이상', '경우', '공무원', '의결', '범위', '단체', '행정', '정당', '보호']


In [28]:
# "법률"이라는 단어가 학습된 100차원 벡터를 확인
print(model.wv.get_vector("법률"))
print(len(model.wv.get_vector("법률")))

[-0.01469253  0.00895459  0.0078085   0.01096     0.01159062 -0.02309667
  0.00691674  0.02582723 -0.01024349 -0.00931482 -0.00838209 -0.02246451
 -0.01043185  0.00949111  0.00734021  0.00362889  0.01151401 -0.00321094
 -0.00580279 -0.02048293  0.00747842 -0.00070232  0.01321437 -0.0183537
  0.00227033  0.00354604 -0.01003395 -0.00329184 -0.01082294  0.00860035
  0.02311528 -0.0020818   0.00534432 -0.01132969 -0.00563503  0.01583404
  0.00415521 -0.0020573   0.00209951 -0.00817892  0.0124905  -0.01764512
 -0.00916473  0.0019083   0.0071561   0.0068926  -0.00278395 -0.0055897
  0.00680099  0.00470069  0.01733286 -0.01534944  0.00264479  0.00352247
 -0.01216361  0.01414101  0.01145648  0.00539523 -0.01711211  0.01087481
 -0.00785353  0.0080934  -0.00379395 -0.010258   -0.00686623  0.0117539
  0.01220504  0.002516   -0.00928068  0.0134455  -0.00997127 -0.00457353
  0.01388325  0.0066944   0.01095384 -0.00105197 -0.00873007 -0.00241322
 -0.00098373 -0.00068528 -0.01532544  0.00514878 -0.00

# 7. 코싸인 유사도 함수 헌법 및 적용

In [30]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

# 코사인 유사도 함수
def cosine_similarity(A, B):
    return dot(A, B) / (norm(A) * norm(B))

# '법률'단어 벡터와 '헌법' 단어 벡터 간의 코사인 유사도 계산
similarity_value = cosine_similarity(model.wv.get_vector('법률'), model.wv.get_vector('헌법')                        )
print("'법률'과 '헌법'의 코사인 유사도:", similarity_value)  # 0.63208294

'법률'과 '헌법'의 코사인 유사도: 0.63208294


In [31]:

# 학습된 벡터 중, '헌법'과 가장 유사도가 높은 순으로 10개 단어를 출력
most_similar_words = model.wv.most_similar('헌법')
print("헌법과 유사한 단어 Top 10:", most_similar_words)

헌법과 유사한 단어 Top 10: [('때', 0.7500783801078796), ('국회', 0.7433488368988037), ('의하', 0.7079368233680725), ('제조', 0.676602840423584), ('공무원', 0.660486102104187), ('대통령', 0.6597691178321838), ('경제', 0.6570447087287903), ('일', 0.6554298400878906), ('관하', 0.6552576422691345), ('위하', 0.6498183012008667)]
